# XE4 FMHA4 forward attention

A **full-execution-flow** functional port of the sycl-tla example [`examples/xe4/fmha4/xe4_fmha_fwd.cpp`](../../examples/xe4/fmha4/xe4_fmha_fwd.cpp ), runnable on the CPU.

Flash-attention forward: S=scale*Q*K^T, softmax, O=P*V — two AMMA GEMMs bridged by an LDSM warp-row softmax.

The cells reproduce the real kernel flow — setup → config → tiling → SLM staging → clear accumulator → prologue → **K-loop mainloop** → epilogue → store → **validation**. Each cell does one operation and uses the shared display helpers from the first code cell. (Same content as `fmha4_fwd_xe4.py`.)

## Helpers — display + SLM copy (reused by every cell below)

In [1]:
import numpy as np
np.set_printoptions(precision=2, suppress=True, linewidth=120)
from tensor_layouts import Layout, size
from tensor_layouts.analysis import is_bijective
from tensor_layouts.atoms_xe_common import make_slm_layout_elem, sizeof_bits

# ---- display helpers (reused by every cell below) ----
def show_mat(name, X, r=4, c=8):
    X = np.asarray(X)
    print(name, " shape", X.shape)
    print(X[:r, :c] if X.ndim == 2 else X[:c])
    if X.ndim == 2 and (X.shape[0] > r or X.shape[1] > c):
        print("   ...(showing %dx%d of %dx%d)" % (min(r, X.shape[0]), min(c, X.shape[1]), *X.shape))

def show_layout(layout, n_rows, n_cols, rl="m", cl="k", max_r=8, max_c=8):
    R, C = min(n_rows, max_r), min(n_cols, max_c)
    print("      " + "".join((cl + str(j)).ljust(5) for j in range(C)) + (" ..." if C < n_cols else ""))
    for i in range(R):
        print((" " + rl + str(i)).ljust(6) + "".join(str(layout(i, j)).ljust(5) for j in range(C))
              + (" ..." if C < n_cols else ""))
    if R < n_rows:
        print("   ...(%dx%d total)" % (n_rows, n_cols))

def check(name, cond):
    print(("[PASS] " if cond else "[FAIL] ") + name)

# ---- SLM staging helpers: gmem tile <-> bank-swizzled SLM buffer ----
def load_to_slm(mat, lay):
    buf = np.zeros(size(lay), dtype=mat.dtype)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            buf[lay(i, j)] = mat[i, j]
    return buf

def read_slm(buf, lay, r, c):
    return np.array([[buf[lay(i, j)] for j in range(c)] for i in range(r)])

print("helpers ready: show_mat, show_layout, check, load_to_slm, read_slm")

helpers ready: show_mat, show_layout, check, load_to_slm, read_slm


## Setup — allocate + initialize operands

In [2]:
# Flash-attention forward. Real tile <64,128,128,128>; small here to print.
seq_q, seq_k, head = 32, 32, 16
BLK_K = 16; K_TILE = head // BLK_K; K_PIPE = 2
scale = 1.0 / np.sqrt(head); DT = "bf16"; DBITS = sizeof_bits(DT)
rng = np.random.default_rng(0)
Q  = rng.integers(-2, 3, size=(seq_q, head)).astype(np.float32)
Kk = rng.integers(-2, 3, size=(seq_k, head)).astype(np.float32)
V  = rng.integers(-2, 3, size=(seq_k, head)).astype(np.float32)
show_mat("Q", Q); show_mat("K", Kk); show_mat("V", V); print("softmax scale:", round(scale, 4))

Q  shape (32, 16)
[[ 2.  1.  0. -1. -1. -2. -2. -2.]
 [ 1.  0.  0.  2. -1.  2.  1. -2.]
 [-2.  2. -2.  0. -2. -1.  0.  0.]
 [-1.  1.  1. -1.  0.  2.  2.  2.]]
   ...(showing 4x8 of 32x16)
K  shape (32, 16)
[[-2.  2.  2. -2.  2.  1. -1.  1.]
 [ 0. -1.  1.  0.  0.  0.  0. -2.]
 [-2. -2. -1.  1. -1.  2. -2.  2.]
 [ 0.  0.  0.  1.  0.  2. -2.  0.]]
   ...(showing 4x8 of 32x16)
V  shape (32, 16)
[[ 2.  2. -1.  1. -2. -1.  0.  2.]
 [ 2. -1.  0.  0. -2. -2.  0. -2.]
 [ 1.  1.  1.  0. -1.  2.  2.  1.]
 [-2.  0.  0.  0. -2. -2. -2.  1.]]
   ...(showing 4x8 of 32x16)
softmax scale: 0.25


## Config — two TiledMMAs (QK, PV) + SLM

In [3]:
from tensor_layouts.atoms_xe4 import make_xe4_amma_atom, make_xe4_ldstm
mmaQK = make_xe4_amma_atom("f32", DT, DT, "f32", seq_q, seq_k, BLK_K, tracking="AB")
mmaPV = make_xe4_amma_atom("f32", DT, DT, "f32", seq_q, head, BLK_K, tracking="AB")
slmQ = make_slm_layout_elem(DBITS, seq_q, head); slmK = make_slm_layout_elem(DBITS, seq_k, head)
print("TiledMmaQK:", mmaQK.name); print("TiledMmaPV:", mmaPV.name, " K-tiles:", K_TILE)

TiledMmaQK: XE4_AMMA_AB_32x32x16_F32BF16BF16F32
TiledMmaPV: XE4_AMMA_AB_32x16x16_F32BF16BF16F32  K-tiles: 1


## ADMA load Q, K into SLM

In [4]:
# ADMA_Q / ADMA_K: stage Q and K through the swizzled SLM.
qbuf = load_to_slm(Q, slmQ)
check("Q SLM round-trip", np.array_equal(read_slm(qbuf, slmQ, seq_q, head), Q))
print("SLM Q layout (row, head) -> offset:"); show_layout(slmQ, seq_q, head, rl="q", cl="d")

[PASS] Q SLM round-trip
SLM Q layout (row, head) -> offset:
      d0   d1   d2   d3   d4   d5   d6   d7    ...
 q0   0    1    2    3    4    5    6    7     ...
 q1   16   17   18   19   20   21   22   23    ...
 q2   128  129  130  131  132  133  134  135   ...
 q3   144  145  146  147  148  149  150  151   ...
 q4   256  257  258  259  260  261  262  263   ...
 q5   272  273  274  275  276  277  278  279   ...
 q6   384  385  386  387  388  389  390  391   ...
 q7   400  401  402  403  404  405  406  407   ...
   ...(32x16 total)


## GEMM-1 mainloop: S = scale*Q*K^T

In [5]:
# GEMM-1 mainloop (TiledMmaQK): S = scale * sum_k Q_k . K_k^T.
S = np.zeros((seq_q, seq_k), np.float32)
for kt in range(K_TILE):
    S += Q[:, kt*BLK_K:(kt+1)*BLK_K] @ Kk[:, kt*BLK_K:(kt+1)*BLK_K].T
S *= scale
show_mat("S = scale * Q . K^T", S)

S = scale * Q . K^T  shape (32, 32)
[[-1.25 -1.5   0.5   0.75  0.25 -0.75 -1.75  2.25]
 [-0.25  0.25  1.    1.    0.    0.    1.25  2.5 ]
 [-1.25  1.75  2.   -2.    0.25 -3.   -1.75 -1.5 ]
 [ 3.75 -3.5   0.25  0.   -0.25 -1.   -0.5   0.  ]]
   ...(showing 4x8 of 32x32)


## Softmax: S -> P

In [6]:
# Row softmax over seq_k keys.
row_max = S.max(1, keepdims=True); P = np.exp(S - row_max); P = P / P.sum(1, keepdims=True)
show_mat("row_max", row_max); show_mat("P = softmax(S)", P); check("rows sum to 1", np.allclose(P.sum(1), 1))

row_max  shape (32, 1)
[[3.25]
 [6.5 ]
 [4.25]
 [3.75]]
   ...(showing 4x1 of 32x1)
P = softmax(S)  shape (32, 32)
[[0.   0.   0.01 0.02 0.01 0.   0.   0.08]
 [0.   0.   0.   0.   0.   0.   0.   0.02]
 [0.   0.03 0.04 0.   0.01 0.   0.   0.  ]
 [0.44 0.   0.01 0.01 0.01 0.   0.01 0.01]]
   ...(showing 4x8 of 32x32)
[PASS] rows sum to 1


## LDSM warp-row (fred.max co-residence)

In [7]:
# LDSM warp-row: a query row's scores co-resident in one 32-lane subgroup.
ldsm = make_xe4_ldstm("LDSM", VS=8, s=DT)
print("LDSM atom:", ldsm.name, " ThrID =", size(ldsm.thr_id))
warp = {t: S[0, t] for t in range(min(seq_k, 32))}
check("in-warp max == numpy", max(warp.values()) == S[0, :min(seq_k, 32)].max())

LDSM atom: XE4_LDSM_VS8_BF16  ThrID = 32
[PASS] in-warp max == numpy


## GEMM-2 mainloop: O = P*V

In [8]:
# GEMM-2 mainloop (TiledMmaPV): O = sum_k P_k . V_k.
O = np.zeros((seq_q, head), np.float32)
for kt in range(seq_k // BLK_K):
    O += P[:, kt*BLK_K:(kt+1)*BLK_K] @ V[kt*BLK_K:(kt+1)*BLK_K, :]
show_mat("O = P . V", O)

O = P . V  shape (32, 16)
[[ 0.56 -0.98  0.17 -0.4   0.07  0.82  0.13 -0.49]
 [ 1.8  -0.75  0.98 -0.94  1.74 -0.1  -0.04 -0.02]
 [ 0.96  0.62 -0.35 -0.39 -0.68  0.44  0.8   0.05]
 [ 0.96  0.68 -0.28  0.34 -0.92 -0.29 -0.44  0.57]]
   ...(showing 4x8 of 32x16)


## Store O

In [9]:
slmO = make_slm_layout_elem(DBITS, seq_q, head); obuf = load_to_slm(O, slmO)
check("O store round-trip", np.array_equal(read_slm(obuf, slmO, seq_q, head), O))

[PASS] O store round-trip


## Reference attention + validation

In [10]:
S_ref = scale * (Q @ Kk.T)
Pr = np.exp(S_ref - S_ref.max(1, keepdims=True)); Pr = Pr / Pr.sum(1, keepdims=True)
O_ref = Pr @ V
err = np.abs(O - O_ref).max(); print("max abs error vs reference attention:", err)
check("FMHA4 verification", err < 1e-5)

max abs error vs reference attention: 3.572222870840136e-07
[PASS] FMHA4 verification


## Recap

load Q,K → QK K-loop → softmax → LDSM → PV K-loop → store → validate.